# FAYE Marie Pierre Gnilane Ndew
# 12213642

# A) LES DONNÉES
## 1) Engendrons un tableau d’observations de la forme suivante

In [1]:
import pandas as pd
import numpy as np
import random
random.seed(0)

lignes = ["BBB", "DDB", "DDB", "DDD", "BBD", "BBD", "BBB", "DDD"]  # 1re lettre de chaque case
Y = ["T"]*5 + ["S"]*3            # 2e composante : dépend de la ligne
Z = ["G", "G", "P"]              # 3e composante : dépend de la colonne

tableau=[]
for i in range(8):
    ligne = []
    for j in range(3):
        case=(lignes[i][j], Y[i],Z[j])
        ligne.append(case)
    tableau.append(ligne)
print(pd.DataFrame(tableau))

           0          1          2
0  (B, T, G)  (B, T, G)  (B, T, P)
1  (D, T, G)  (D, T, G)  (B, T, P)
2  (D, T, G)  (D, T, G)  (B, T, P)
3  (D, T, G)  (D, T, G)  (D, T, P)
4  (B, T, G)  (B, T, G)  (D, T, P)
5  (B, S, G)  (B, S, G)  (D, S, P)
6  (B, S, G)  (B, S, G)  (B, S, P)
7  (D, S, G)  (D, S, G)  (D, S, P)


## 2) Ecrivons une fonction Python qui nous permettent d’engendrer d’une manière aléatoire en utilisant une distribution uniforme un tableau de la même forme, mais avec n rangées et 3 colonnes

In [2]:
def engendrer_tableau(n):
    tableau1 = []
    for i in range(n):
        ligne1=[]
        for j in range(3):
            y = random.choice(["T","S"])
            a = random.choice(["B","D"])
            b = random.choice(["G","P"])
            ligne1.append((a,y,b))
        tableau1.append(ligne1)
    return tableau1

t = engendrer_tableau(8)
t

[[('D', 'S', 'G'), ('D', 'S', 'P'), ('D', 'S', 'P')],
 [('B', 'T', 'P'), ('B', 'T', 'P'), ('D', 'T', 'G')],
 [('D', 'T', 'P'), ('D', 'T', 'P'), ('B', 'S', 'P')],
 [('D', 'S', 'G'), ('B', 'T', 'P'), ('D', 'T', 'P')],
 [('D', 'T', 'G'), ('B', 'T', 'G'), ('D', 'T', 'G')],
 [('D', 'T', 'P'), ('D', 'T', 'P'), ('D', 'T', 'G')],
 [('D', 'S', 'G'), ('D', 'S', 'G'), ('B', 'S', 'G')],
 [('B', 'T', 'P'), ('B', 'S', 'G'), ('B', 'T', 'G')]]

## 3) On suppose que l’ensemble de données ainsi obtenu décrit les trois variables suivantes : Cheveux = B, D, où B=blond, D=dark. Hauteur = T, S, où T=tall,S=short. Pays = G, P, où G=Greenland, P=Poland

# B)QUESTIONS
Nous voulons répondre à la question suivante : si nous observons un nouvel individu grand ayant cheveux blonds, quel est son pays d'origine le plus probable?
## B-1) Remarques:
Il s'agit d'écrire une fonction Python qui engendre une matrice ayant N lignes dont chaque élément est constitué d'une triplet du type :$(X_1, X_2, X_3), \quad X_i \neq X_j, \quad X_i \in \{D, T, G\}$

## B-2) Codes:

In [3]:
H=engendrer_tableau(1000)
individus = []
for ligne in H:
    for case in ligne:
        individus.append(case)

df = pd.DataFrame(individus, columns=["Cheveux","Hauteur", "Pays"])
grand_blonds = df[(df["Cheveux"]=="B") & (df["Hauteur"]=="T")]
print(len(grand_blonds))
print(grand_blonds["Pays"].value_counts(normalize=True))

760
Pays
P    0.517105
G    0.482895
Name: proportion, dtype: float64


## Vérification

In [4]:
resultats = []
for essai in range(20):
    H = engendrer_tableau(1000)
    individus = [case for ligne in H for case in ligne]
    df = pd.DataFrame(individus, columns=["Cheveux","Hauteur","Pays"])
    gb = df[(df["Cheveux"]=="B") & (df["Hauteur"]=="T")]
    resultats.append(gb["Pays"].value_counts(normalize=True)["G"])

print(sum(1 for r in resultats if r > 0.5), "fois sur 20 en faveur de G")

12 fois sur 20 en faveur de G


`engendrer_tableau(n)` engendre `n` rangées de 3 triplets `(Cheveux, Hauteur, Pays)`, chaque composante tirée uniformément et indépendamment (probabilité 1/2 par valeur).
On aplatit la matrice en `3n` individus, on garde les grands blonds (B et T), et on calcule la fréquence de chaque pays : ce sont les estimations de $P(\text{Pays} \mid B, T)$.
Avec `n = 1000` : 760 grands blonds, 48,3 % G et 51,7 % P. L'écart n'est pas significatif : en répétant l'expérience, G l'emporte 12 fois sur 20, donc le pays gagnant change d'un tirage à l'autre.
Conclusion : aucun pays n'est plus probable, $P(\text{Pays} \mid B, T) = P(\text{Pays}) = 1/2$, car le pays est tiré indépendamment des deux autres variables.

# C) Maximum a posteriori
## C-1) Écriture d'une fonction python qui donnera la solution MAP en utilisant le cadre naÏf de BAYES

In [5]:
def map_naif(df, cheveux="B", hauteur="T"):
    scores = {}
    for pays in ["G", "P"]:
        sub = df[df["Pays"] == pays]
        p_pays = len(sub) / len(df) # P(pays)
        p_c = (sub["Cheveux"] == cheveux).mean() # P(B|pays)
        p_h = (sub["Hauteur"] == hauteur).mean() # P(T|pays)
        scores[pays] = p_c * p_h * p_pays
    return max(scores, key=scores.get), scores

lignes = ["BBB", "DDB", "DDB", "DDD", "BBD", "BBD", "BBB", "DDD"]
Y = ["T"]*5 + ["S"]*3
Z = ["G", "G", "P"]
individus_ex = [(lignes[i][j], Y[i], Z[j]) for i in range(8) for j in range(3)]
df_ex = pd.DataFrame(individus_ex, columns=["Cheveux", "Hauteur", "Pays"])
print(df_ex.head())

map_naif(df_ex)

  Cheveux Hauteur Pays
0       B       T    G
1       B       T    G
2       B       T    P
3       D       T    G
4       D       T    G


('G',
 {'G': np.float64(0.20833333333333331), 'P': np.float64(0.10416666666666666)})

## C-2) Écriture d'une fonction python qui donnera la solution MAP sans utiliser le cadre naÏf de BAYES

In [6]:
def map_direct(df, cheveux="B", hauteur="T"):
    scores = {}
    for pays in ["G", "P"]:
        sub = df[df["Pays"] == pays]
        p_pays = len(sub) / len(df)                      # P(pays)
        p_ct = ((sub["Cheveux"] == cheveux) &
                (sub["Hauteur"] == hauteur)).mean()      # P(B,T|pays)
        scores[pays] = p_ct * p_pays
    return max(scores, key=scores.get), scores

map_direct(df_ex)

('G', {'G': np.float64(0.16666666666666666), 'P': np.float64(0.125)})

Les deux méthodes désignent Greenland comme pays le plus probable pour un individu grand et blond. Sous l'hypothèse naïve, $P(G|B,T) = 2/3$ ; sans cette hypothèse, $P(G|B,T) = 4/7$, valeur exacte que l'on retrouve en comptant directement les 7 grands blonds du tableau, dont 4 viennent de Greenland. L'écart entre les deux mesure le coût de l'hypothèse d'indépendance conditionnelle, ici légèrement fausse puisque $P(B,T|G) = 1/4 \neq P(B|G)P(T|G) = 5/16$.

# D) Maximum de vraisemblance
## 1) Ecriture d'une fonction Python qui donnera la solution de maximum de vraisemblance (MLE) à la question, en utilisant le cadre naÏf de BAYES.

In [7]:
def mle_naif(df, cheveux="B", hauteur="T"):
    scores = {}
    for pays in ["G", "P"]:
        sub = df[df["Pays"] == pays]
        p_c = (sub["Cheveux"] == cheveux).mean()     # P(B|pays)
        p_h = (sub["Hauteur"] == hauteur).mean()     # P(T|pays)
        scores[pays] = p_c * p_h
    return max(scores, key=scores.get), scores

mle_naif(df_ex)

('G', {'G': np.float64(0.3125), 'P': np.float64(0.3125)})

## 2) Ecriture d'une fonction Python qui donnera la solution de maximum de vraisemblance (MLE) à la question, sans utiliser le cadre naÏf de BAYES.

In [8]:
def mle_direct(df, cheveux="B", hauteur="T"):
    scores = {}
    for pays in ["G", "P"]:
        sub = df[df["Pays"] == pays]
        scores[pays] = ((sub["Cheveux"] == cheveux) &
                        (sub["Hauteur"] == hauteur)).mean()   # P(B,T|pays)
    return max(scores, key=scores.get), scores

mle_direct(df_ex)

('P', {'G': np.float64(0.25), 'P': np.float64(0.375)})

Le MLE ne compare que les vraisemblances $P(B,T|G)$ et $P(B,T|P)$, sans tenir compte des probabilités a priori. Sous l'hypothèse naïve, les deux valent $\frac{8}{16} \cdot \frac{10}{16} = \frac{5}{16}$ : les deux pays sont à égalité et le MLE ne tranche pas (la fonction renvoie G par simple effet de l'ordre des clés). Sans cette hypothèse, on obtient $P(B,T|G) = 2/8$ et $P(B,T|P) = 3/8$, donc le MLE désigne Poland.
Ce résultat s'oppose à celui du MAP, qui désignait Greenland. La différence vient des a priori : les Groenlandais représentent 2/3 des individus du tableau, et en multipliant les vraisemblances par $P(G) = 2/3$ et $P(P) = 1/3$ on renverse la comparaison, $\frac28 \cdot \frac23 = \frac16 > \frac38 \cdot \frac13 = \frac18$. Le MLE répond à « dans quel pays ce profil est-il le plus typique ? », le MAP à « d'où vient le plus probablement cet individu ? ».

# E)Point de vue continu
## 1) Engendrons un tel jeu de données via une fonction Python

In [9]:
def engendrer_continu(N):
    donnees = []
    for i in range(N):
        pays = random.choice(["G", "P"])
        hauteur = random.choice(["T", "S"])
        mu = 0.35 if pays == "G" else 0.65      # moyenne d'intensité selon le pays
        x = min(max(random.gauss(mu, 0.15), 0.001), 0.999)
        donnees.append((x, hauteur, pays))
    return pd.DataFrame(donnees, columns=["Intensite", "Hauteur", "Pays"])

df_c = engendrer_continu(2000)
print(df_c.head())
print(df_c.groupby("Pays")["Intensite"].mean())

   Intensite Hauteur Pays
0   0.745809       S    P
1   0.720805       T    P
2   0.531544       S    P
3   0.243070       T    G
4   0.273597       S    G
Pays
G    0.352216
P    0.649398
Name: Intensite, dtype: float64


## 2)
Le problème est que « blond » n'est plus un événement que l'on peut compter. Demander $P(\text{Intensite} = 0{,}2 \mid G)$ n'a pas de sens, car une variable continue a une probabilité nulle de valoir exactement une valeur donnée. On remplace donc la probabilité par une densité $f(x \mid G)$, et on compare
$$f(x \mid G)\,P(T \mid G)\,P(G) \quad \text{contre} \quad f(x \mid P)\,P(T \mid P)\,P(P)$$
Le raisonnement de Bayes reste identique, seul le terme des cheveux change de nature. Deux façons d'estimer cette densité : supposer une forme paramétrique, typiquement une gaussienne dont on estime moyenne et écart-type sur chaque pays (c'est le Gaussian Naive Bayes) ; ou discrétiser l'intensité en classes (par exemple $[0;0{,}5)$ = blond, $[0{,}5;1)$ = dark) et retomber exactement sur le cas précédent.

## 3) Modifications des fonctions

In [10]:
from scipy.stats import norm

def map_gaussien(df, x, hauteur="T"):
    scores = {}
    for pays in ["G", "P"]:
        sub = df[df["Pays"] == pays]
        p_pays = len(sub) / len(df)
        f_x = norm.pdf(x, sub["Intensite"].mean(), sub["Intensite"].std())  # densité
        p_h = (sub["Hauteur"] == hauteur).mean()
        scores[pays] = f_x * p_h * p_pays
    return max(scores, key=scores.get), scores

print(map_gaussien(df_c, 0.2))   # individu très blond et grand
print(map_gaussien(df_c, 0.8))   # individu très foncé et grand

('G', {'G': np.float64(0.38722267850571435), 'P': np.float64(0.0057343087697159035)})
('P', {'G': np.float64(0.006105322617986927), 'P': np.float64(0.3732663577916959)})


La seule modification consiste à remplacer l'estimation de $P(B \mid \text{Pays})$ par comptage — impossible pour une variable continue — par une densité gaussienne $f(x \mid \text{Pays})$, dont la moyenne et l'écart-type sont estimés sur chaque pays. Le reste des fonctions est inchangé ; pour le MLE il suffit de retirer le facteur $P(\text{Pays})$. La réponse dépend désormais de l'intensité observée : G pour $x = 0{,}2$, P pour $x = 0{,}8$, avec une frontière de décision autour de $x = 0{,}5$. Les scores ne sont plus des probabilités puisqu'ils contiennent une densité, mais la comparaison reste valide ; après normalisation on retrouve des probabilités a posteriori.